# Bot!

Bot is basically ones that deduct the correct pairs based on the benchmarking information. It only knows the graph (`CouplingMap` object), the data (from `mi`), and its identity. Philosophical, right?

First, run the experiment as usual.

In [1]:
import numpy as np
import matplotlib.pyplot as plt
from qiskit_aer import AerSimulator
from qiskit_aer.noise import NoiseModel, depolarizing_error
from qiskit.transpiler import Target, CouplingMap
from qiskit.quantum_info import Operator
from qiskit.circuit.library import CXGate
from qiskit_device_benchmarking.bench_code.mrb import MirrorQA, QuantumAwesomeness
import os, random, json

SEED = 123
os.environ["PYTHONHASHSEED"] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)

In [2]:
basis_gates = ['id', 'h', 'x', 'y', 'z', 'rz', 'cx']
p2 = 1e-1
p1 = p2 / 10
rz_angle = np.pi / 2

shots = 1000
num_samples = 20
lengths = [2] + [4, 10, 20, 50]

# This can be changed in the future when we want to upgrade the square lattice or use the real IBM machine which has the square lattice.
num_qubits = 16
cmap = CouplingMap.from_grid(4, 4, bidirectional=True)

# Set up the target object (exclusively designed RZ gate)
target = Target.from_configuration(
    num_qubits=num_qubits,
    basis_gates=basis_gates,
    coupling_map=cmap,
    custom_name_mapping={
        "id": Operator(np.array([[1, 0], [0, 1]])),  # Identity gate
        "h": Operator(np.array([[1, 1], [1, -1]]) / np.sqrt(2)),  # Hadamard gate
        "x": Operator(np.array([[0, 1], [1, 0]])),  # Pauli X gate
        "y": Operator(np.array([[0, -1j], [1j, 0]])),  # Pauli Y gate
        "z": Operator(np.array([[1, 0], [0, -1]])),  # Pauli Z gate
        "rz": Operator(
            [
                [np.cos(rz_angle / 2), -1j * np.sin(rz_angle / 2)],
                [-1j * np.sin(rz_angle / 2), np.cos(rz_angle / 2)],
            ]
        ),  # RZ(rz_angle) from above
        "cx": Operator(
            np.array([[1, 0, 0, 0], [0, 1, 0, 0], [0, 0, 0, 1], [0, 0, 1, 0]])
        ),  # CNOT gate
    },
)

# Set up the noise model
noise_model = NoiseModel()
error_1q = depolarizing_error(p1, 1)
error_2q = depolarizing_error(p2, 2)
for gate in basis_gates:
    if gate in ['id', 'h', 'x', 'y', 'z', 'rz']:
        noise_model.add_all_qubit_quantum_error(error_1q, gate)
    elif gate == 'cx':
        noise_model.add_all_qubit_quantum_error(error_2q, gate)

# Set up the simulation backend (or, in the future, IBM device)       
backend = AerSimulator(
    method="stabilizer",
    noise_model = noise_model,
    target=target,
    max_parallel_threads=0,
    max_parallel_experiments=0,
    seed_simulator=SEED,
)

In [3]:
# The main object to play with
exp = MirrorQA(
    range(num_qubits),
    lengths=lengths,
    backend=backend,
    num_samples=num_samples,
    initial_entangling_angle=np.pi / 2,
    sampling_algorithm='new',
    seed=SEED,
)

exp.set_run_options(shots=shots)

Run the experiment.

In [4]:
rb_data = exp.run()
print("Done.", rb_data.job_ids)

Done. ['84f6bc61-ab09-4433-b08b-35fccc51f5a5']


In the future, we will use other topologically-detailed coupling map but let's now focus on `NewSampler`. First step, we will make all possible pairs connected.

Steps of making a bot: Coupling map -> MI data -> Bot

Thus, we need to extract MI data.

In [6]:
# First, run this
exp.analysis.set_options(analyzed_quantity="Effective Polarization")
# exp.analysis.set_options(analyzed_quantity='Mutual Information')
analysis = exp.analysis.run(rb_data)

In [7]:
# Calculate the MI.
qa = QuantumAwesomeness(exp.backend.coupling_map)
mi = qa.mutual_info(rb_data.data())  # Calculate the MIs per each pair


In [8]:
mi

[{(0, 4): np.float64(0.47952539320307574),
  (0, 1): np.float64(0.0012743066622653565),
  (1, 5): np.float64(0.5270108953269541),
  (1, 2): np.float64(0.002501317755316035),
  (2, 6): np.float64(3.261926946329652e-07),
  (2, 3): np.float64(0.4485887447473388),
  (3, 7): np.float64(0.00010389149823608612),
  (4, 8): np.float64(3.218442519048459e-06),
  (4, 5): np.float64(0.0010782732302024867),
  (5, 9): np.float64(0.0006181978307201463),
  (5, 6): np.float64(0.0005183129292659627),
  (6, 10): np.float64(0.003654486673278501),
  (6, 7): np.float64(0.46495524736619254),
  (7, 11): np.float64(0.0005973310815228228),
  (8, 12): np.float64(0.5185291746218383),
  (8, 9): np.float64(3.6225053290372955e-08),
  (9, 13): np.float64(0.45686988731884093),
  (9, 10): np.float64(0.00012098095580193036),
  (10, 14): np.float64(2.1440883003487343e-05),
  (10, 11): np.float64(0.5397673870181574),
  (11, 15): np.float64(5.505738678701633e-05),
  (12, 13): np.float64(0.00014208506391510944),
  (13, 14): 

In [9]:
import networkx as nx

In [10]:
for i, info in enumerate(mi): # direction doesn't matter in mi but from the coupling map didn' we set the direction?
    print(i, info)

0 {(0, 4): np.float64(0.47952539320307574), (0, 1): np.float64(0.0012743066622653565), (1, 5): np.float64(0.5270108953269541), (1, 2): np.float64(0.002501317755316035), (2, 6): np.float64(3.261926946329652e-07), (2, 3): np.float64(0.4485887447473388), (3, 7): np.float64(0.00010389149823608612), (4, 8): np.float64(3.218442519048459e-06), (4, 5): np.float64(0.0010782732302024867), (5, 9): np.float64(0.0006181978307201463), (5, 6): np.float64(0.0005183129292659627), (6, 10): np.float64(0.003654486673278501), (6, 7): np.float64(0.46495524736619254), (7, 11): np.float64(0.0005973310815228228), (8, 12): np.float64(0.5185291746218383), (8, 9): np.float64(3.6225053290372955e-08), (9, 13): np.float64(0.45686988731884093), (9, 10): np.float64(0.00012098095580193036), (10, 14): np.float64(2.1440883003487343e-05), (10, 11): np.float64(0.5397673870181574), (11, 15): np.float64(5.505738678701633e-05), (12, 13): np.float64(0.00014208506391510944), (13, 14): np.float64(1.3035060322241776e-06), (14, 15

In [11]:
for i, info in enumerate(mi):
    G = nx.Graph()
    for (prev, next), w in info.items():
        G.add_edge(prev, next, weight=w)
        
    guess = nx.max_weight_matching(G, maxcardinality=True, weight='weight')
    truth = set(tuple(sorted(p)) for p in exp._pairs[i])
    sorted_guess = set(tuple(sorted(a)) for a in guess)
        
    yesss = (sorted_guess == truth)
    print(f"Circuit {i:3d}: {'YES' if yesss else 'NO'}  outermost={sorted(exp._pairs[i])}  bot={sorted_guess}")

Circuit   0: YES  outermost=[(3, 2), (4, 0), (5, 1), (7, 6), (8, 12), (10, 11), (13, 9), (14, 15)]  bot={(10, 11), (9, 13), (0, 4), (1, 5), (2, 3), (6, 7), (8, 12), (14, 15)}
Circuit   1: YES  outermost=[(3, 2), (4, 0), (5, 1), (7, 6), (8, 12), (10, 11), (13, 9), (14, 15)]  bot={(10, 11), (9, 13), (0, 4), (1, 5), (2, 3), (6, 7), (8, 12), (14, 15)}
Circuit   2: YES  outermost=[(1, 2), (4, 0), (5, 9), (6, 10), (7, 3), (8, 12), (13, 14), (15, 11)]  bot={(13, 14), (1, 2), (0, 4), (3, 7), (11, 15), (6, 10), (8, 12), (5, 9)}
Circuit   3: YES  outermost=[(0, 1), (3, 7), (5, 4), (6, 2), (8, 12), (9, 10), (11, 15), (13, 14)]  bot={(9, 10), (13, 14), (0, 1), (3, 7), (4, 5), (2, 6), (11, 15), (8, 12)}
Circuit   4: NO  outermost=[(1, 0), (2, 6), (3, 7), (4, 8), (9, 5), (10, 11), (12, 13), (15, 14)]  bot={(0, 1), (10, 11), (12, 13), (2, 3), (6, 7), (4, 8), (5, 9), (14, 15)}
Circuit   5: YES  outermost=[(0, 4), (1, 5), (2, 3), (6, 7), (8, 9), (10, 14), (11, 15), (12, 13)]  bot={(0, 4), (1, 5), (10, 